In [1]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms 
from torchvision import models
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


## Make a Fine Tuning in model 

we dont change the layers  and we just add the output layer to classfiy the 10 

- name is the Head Fine-tuning 

In [5]:


head_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

print(head_model.layer4)

for param in head_model.parameters():
    param.requires_grad = False
    
head_model.fc = nn.Linear(head_model.fc.in_features, 10)

Sequential(
  (0): BasicBlock(
    (conv1): Conv2d(256, 512, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (conv2): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn2): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (downsample): Sequential(
      (0): Conv2d(256, 512, kernel_size=(1, 1), stride=(2, 2), bias=False)
      (1): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    )
  )
  (1): BasicBlock(
    (conv1): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn1): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (conv2): Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), b

## Now in this step we make a fine tuning in the last 4 layer


- name is Partial Fine-tuning

In [3]:


partial_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Freeze everything
for param in partial_model.parameters():
    param.requires_grad = False

# Replace the classifier
partial_model.fc = nn.Linear(partial_model.fc.in_features, 10)

# Unfreeze the last residual block
for param in partial_model.layer4.parameters():
    param.requires_grad = True

n = 0
    
for name, param in partial_model.named_parameters():
    if param.requires_grad:
        n += 1
    # print(name, param.requires_grad)
print(f"Number of trainable parameters: {n}")

Number of trainable parameters: 17


In [4]:
trainable_tensors = sum(
    1 for p in partial_model.parameters()
    if p.requires_grad
)

trainable_weights = sum(
    p.numel() for p in partial_model.parameters()
    if p.requires_grad
)

total_weights = sum(
    p.numel() for p in partial_model.parameters()
)

print("Trainable tensors:", trainable_tensors)
print("Trainable weights:", trainable_weights)
print("Total weights:", total_weights)
print(
    f"Trainable percentage: "
    f"{100 * trainable_weights / total_weights:.2f}%"
)




for name, param in partial_model.named_parameters():
    if param.requires_grad:
        print(
            f"{name:40} "
            f"shape={str(tuple(param.shape)):20} "
            f"numel={param.numel():>10,}"
        )

Trainable tensors: 17
Trainable weights: 8398858
Total weights: 11181642
Trainable percentage: 75.11%
layer4.0.conv1.weight                    shape=(512, 256, 3, 3)     numel= 1,179,648
layer4.0.bn1.weight                      shape=(512,)               numel=       512
layer4.0.bn1.bias                        shape=(512,)               numel=       512
layer4.0.conv2.weight                    shape=(512, 512, 3, 3)     numel= 2,359,296
layer4.0.bn2.weight                      shape=(512,)               numel=       512
layer4.0.bn2.bias                        shape=(512,)               numel=       512
layer4.0.downsample.0.weight             shape=(512, 256, 1, 1)     numel=   131,072
layer4.0.downsample.1.weight             shape=(512,)               numel=       512
layer4.0.downsample.1.bias               shape=(512,)               numel=       512
layer4.1.conv1.weight                    shape=(512, 512, 3, 3)     numel= 2,359,296
layer4.1.bn1.weight                      shape=(

In [12]:
full_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

for param in full_model.parameters():
    param.requires_grad = True

n = 0
    
for name, param in full_model.named_parameters():
    if param.requires_grad:
        n += 1
    print(name, param.requires_grad)
print(f"Number of trainable parameters: {n}")

conv1.weight True
bn1.weight True
bn1.bias True
layer1.0.conv1.weight True
layer1.0.bn1.weight True
layer1.0.bn1.bias True
layer1.0.conv2.weight True
layer1.0.bn2.weight True
layer1.0.bn2.bias True
layer1.1.conv1.weight True
layer1.1.bn1.weight True
layer1.1.bn1.bias True
layer1.1.conv2.weight True
layer1.1.bn2.weight True
layer1.1.bn2.bias True
layer2.0.conv1.weight True
layer2.0.bn1.weight True
layer2.0.bn1.bias True
layer2.0.conv2.weight True
layer2.0.bn2.weight True
layer2.0.bn2.bias True
layer2.0.downsample.0.weight True
layer2.0.downsample.1.weight True
layer2.0.downsample.1.bias True
layer2.1.conv1.weight True
layer2.1.bn1.weight True
layer2.1.bn1.bias True
layer2.1.conv2.weight True
layer2.1.bn2.weight True
layer2.1.bn2.bias True
layer3.0.conv1.weight True
layer3.0.bn1.weight True
layer3.0.bn1.bias True
layer3.0.conv2.weight True
layer3.0.bn2.weight True
layer3.0.bn2.bias True
layer3.0.downsample.0.weight True
layer3.0.downsample.1.weight True
layer3.0.downsample.1.bias True
la

. Feature Extraction
تدريب الـ Classifier فقط
- Backbone: مجمّد Frozen.
- fc: قابل للتدريب.
- أسرع وأقل استهلاكًا للموارد.



1. Partial Fine-tuning
تدريب أجزاء مختارة
- layer1 وlayer2 وlayer3: مجمّدة في مثالنا.
- layer4 وfc: قابلان للتدريب.
- توازن بين سرعة التدريب والتكيف مع البيانات الجديدة.


1. Full Fine-tuning
تدريب النموذج بالكامل
- جميع طبقات الـ Backbone قابلة للتدريب.
- fc قابلة للتدريب.
- مرونة أكبر، لكنها قد تحتاج وقتًا وموارد أكثر.

In [17]:
print(f"Trainable weights: {sum(p.numel() for p in partial_model.parameters() if p.requires_grad):,}")

Trainable weights: 8,398,858
